# Lomb–Scargleによる呼吸数推定

このノートブックではCSIファイルの読込・振幅化・サブキャリア選択・PCAを再実装しません。既存のCSI解析パイプラインが生成した`principal_components`と、同じパケットに対応する`timestamps_ns`を受け取り、Lomb–Scargle解析だけを実行します。実運用では、アップロードまたはバックグラウンド取得されたCSIがこの共通パイプラインへ渡されます。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lombscargle

BPM_MIN = 6
BPM_MAX = 40
FREQ_MIN_HZ = 0.05
FREQ_MAX_HZ = 1.5
N_FREQUENCIES = 1000


def estimate_breathing_rate_by_lombscargle(
    principal_components,
    timestamps_ns,
    bpm_min=BPM_MIN,
    bpm_max=BPM_MAX,
    show_plot=True,
):
    """既存パイプラインのPCA出力を不均一タイムスタンプで解析する。"""
    components = np.asarray(principal_components, dtype=np.float64)
    timestamps = np.asarray(timestamps_ns, dtype=np.int64).reshape(-1)
    if components.ndim != 2 or components.shape[0] != timestamps.size:
        raise ValueError("PCA出力の行数とタイムスタンプ数が一致していません")
    if components.shape[0] < 4 or components.shape[1] < 1:
        raise ValueError("Lomb–Scargle解析に必要なPCA出力が不足しています")

    # PCA行と時刻の対応を保ったまま並べ替え、重複時刻を除去する。
    order = np.argsort(timestamps, kind="stable")
    timestamps = timestamps[order]
    components = components[order]
    keep = np.concatenate(([True], np.diff(timestamps) > 0))
    timestamps = timestamps[keep]
    components = components[keep]

    time_seconds = (timestamps - timestamps[0]).astype(np.float64) / 1e9
    frequencies_hz = np.linspace(FREQ_MIN_HZ, FREQ_MAX_HZ, N_FREQUENCIES)
    frequencies_bpm = frequencies_hz * 60
    angular_frequencies = 2 * np.pi * frequencies_hz

    periodograms = np.column_stack([
        lombscargle(
            time_seconds,
            components[:, index],
            angular_frequencies,
            precenter=True,
            normalize=True,
            floating_mean=True,
        )
        for index in range(components.shape[1])
    ])

    valid_mask = (frequencies_bpm >= bpm_min) & (frequencies_bpm <= bpm_max)
    peak_bins = []
    peak_powers = []
    for index in range(components.shape[1]):
        local_bin = int(np.argmax(periodograms[valid_mask, index]))
        global_bin = int(np.flatnonzero(valid_mask)[local_bin])
        peak_bins.append(global_bin)
        peak_powers.append(float(periodograms[global_bin, index]))

    selected_pc = int(np.argmax(peak_powers))
    estimated_bin = peak_bins[selected_pc]
    estimated_bpm = float(frequencies_bpm[estimated_bin])

    if show_plot:
        plt.figure(figsize=(12, 5))
        for index in range(components.shape[1]):
            selected = index == selected_pc
            plt.plot(
                frequencies_bpm,
                periodograms[:, index],
                label=f"PC{index + 1}" + (" (Selected)" if selected else ""),
                alpha=1.0 if selected else 0.4,
                linewidth=2.5 if selected else 1.2,
            )
        plt.axvline(estimated_bpm, color="red", linestyle="--", label=f"{estimated_bpm:.2f} bpm")
        plt.axvspan(bpm_min, bpm_max, alpha=0.1, color="green", label="Breathing Range")
        plt.xlabel("Frequency [bpm]")
        plt.ylabel("Power (Normalized)")
        plt.xlim(0, 60)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

    return {
        "breathing_rate_bpm": estimated_bpm,
        "selected_pc": selected_pc + 1,
        "peak_power": peak_powers[selected_pc],
        "frequencies_bpm": frequencies_bpm,
        "periodograms": periodograms,
    }

In [ ]:
# 実運用では既存CSI解析パイプラインから次の2変数を受け取ります。
# result = estimate_breathing_rate_by_lombscargle(
#     principal_components=principal_components,
#     timestamps_ns=timestamps_ns,
# )
# print(f"推定呼吸数: {result['breathing_rate_bpm']:.2f} bpm")